# Bati Bank Credit Risk — Exploratory Data Analysis

This notebook explores the **Xente transaction dataset** to understand customer behavior, transaction patterns, and proxy risk signals (e.g. fraud flags) ahead of model development.

> **Note:** This notebook is for exploration only. Reusable logic lives in `src/data_processing.py` and `src/eda.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Allow imports from project root when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    AMOUNT_COLUMN,
    CUSTOMER_ID_COLUMN,
    PROXY_TARGET_COLUMN,
    VALUE_COLUMN,
)
from src.data_processing import load_transactions
from src.eda import (
    compute_risk_metrics,
    customer_risk_profile,
    dataset_overview,
    describe_categorical,
    describe_numeric,
    group_summary,
    missingness_report,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)

## 1. Load Raw Data

In [ ]:
# Load full dataset; use nrows=10_000 for faster iteration during development
df = load_transactions(parse_dates=True)
df.head()

## 2. Dataset Structure & Types

In [ ]:
overview = dataset_overview(df)
print(f"Rows: {overview['rows']:,}")
print(f"Columns: {overview['columns']}")
print(f"Memory: {overview['memory_mb']} MB")
print(f"Duplicate rows: {overview['duplicate_rows']:,}")
print(f"\nNumeric columns: {overview['numeric_columns']}")
print(f"Categorical columns: {overview['categorical_columns']}")

pd.Series(overview["dtypes"], name="dtype").to_frame()

In [ ]:
df.info()

## 3. Missing Values

In [ ]:
missing = missingness_report(df)
missing

In [ ]:
missing_with_gaps = missing[missing["missing_count"] > 0]
if missing_with_gaps.empty:
    print("No missing values detected in any column.")
else:
    ax = missing_with_gaps.plot.bar(x="column", y="missing_pct", legend=False, color="coral")
    ax.set_title("Missing Values by Column (%)")
    ax.set_ylabel("Missing %")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 4. Summary Statistics

In [ ]:
numeric_summary = describe_numeric(df)
numeric_summary

In [ ]:
risk_metrics = compute_risk_metrics(df)
pd.Series(risk_metrics, name="value").to_frame()

In [ ]:
cat_cols = [
    "ProductCategory",
    "ChannelId",
    "CurrencyCode",
    "PricingStrategy",
    PROXY_TARGET_COLUMN,
]
cat_cols = [c for c in cat_cols if c in df.columns]

categorical_summaries = describe_categorical(df, columns=cat_cols, top_n=8)
for col, summary in categorical_summaries.items():
    print(f"\n=== {col} (unique: {summary.attrs['unique_count']}) ===")
    display(summary)

## 5. Numerical Feature Distributions

In [ ]:
num_features = [AMOUNT_COLUMN, VALUE_COLUMN, "PricingStrategy"]
num_features = [c for c in num_features if c in df.columns]

fig, axes = plt.subplots(1, len(num_features), figsize=(5 * len(num_features), 4))
if len(num_features) == 1:
    axes = [axes]

for ax, col in zip(axes, num_features):
    sns.histplot(df[col], bins=50, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Log-scale view for heavy-tailed monetary features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, [AMOUNT_COLUMN, VALUE_COLUMN]):
    positive = df[df[col] > 0][col]
    sns.histplot(positive, bins=50, log_scale=True, ax=ax)
    ax.set_title(f"Log-scale distribution: {col} (> 0)")

plt.tight_layout()
plt.show()

## 6. Categorical Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

plot_cols = ["ProductCategory", "ChannelId", "PricingStrategy", PROXY_TARGET_COLUMN]
plot_cols = [c for c in plot_cols if c in df.columns]

for ax, col in zip(axes, plot_cols):
    order = df[col].value_counts().head(10).index
    sns.countplot(data=df, y=col, order=order, ax=ax)
    ax.set_title(f"Top categories: {col}")

for ax in axes[len(plot_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Correlations Between Numerical Variables

In [ ]:
numeric_for_corr = df.select_dtypes(include="number").drop(
    columns=["CountryCode"], errors="ignore"
)
corr = numeric_for_corr.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix — Numerical Features")
plt.tight_layout()
plt.show()

corr

## 8. Outlier Detection (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, y=AMOUNT_COLUMN, ax=axes[0])
axes[0].set_title(f"Box plot: {AMOUNT_COLUMN}")

sns.boxplot(data=df, y=VALUE_COLUMN, ax=axes[1])
axes[1].set_title(f"Box plot: {VALUE_COLUMN}")

plt.tight_layout()
plt.show()

In [ ]:
# Box plots by category help reveal segment-specific outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="ProductCategory", y=VALUE_COLUMN, ax=axes[0])
axes[0].set_title("Transaction Value by Product Category")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(data=df, x=PROXY_TARGET_COLUMN, y=VALUE_COLUMN, ax=axes[1])
axes[1].set_title("Transaction Value by Fraud Flag")

plt.tight_layout()
plt.show()

## 9. Patterns by Customer

In [ ]:
customer_profile = customer_risk_profile(df)
customer_profile.describe(include="all").T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(customer_profile["transaction_count"], bins=40, ax=axes[0])
axes[0].set_title("Transactions per Customer")
axes[0].set_xlabel("Transaction count")

sns.histplot(customer_profile["total_value"], bins=40, log_scale=True, ax=axes[1])
axes[1].set_title("Total Value per Customer (log scale)")

plt.tight_layout()
plt.show()

print(f"Customers with proxy event: {customer_profile['has_proxy_event'].sum()}")
print(f"Customer-level proxy rate: {customer_profile['has_proxy_event'].mean():.2%}")

## 10. Patterns by Product Category

In [ ]:
product_summary = group_summary(
    df,
    by="ProductCategory",
    value_cols=[VALUE_COLUMN, PROXY_TARGET_COLUMN],
    agg_funcs=["count", "mean", "sum"],
)
product_summary.sort_values(f"{VALUE_COLUMN}_sum", ascending=False)

In [ ]:
product_fraud = (
    df.groupby("ProductCategory")[PROXY_TARGET_COLUMN]
    .agg(proxy_rate="mean", transactions="count")
    .sort_values("proxy_rate", ascending=False)
)
product_fraud

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(
    data=product_fraud.reset_index(),
    x="ProductCategory",
    y="proxy_rate",
    ax=ax,
)
ax.set_title("Proxy Event Rate by Product Category")
ax.set_ylabel("Fraud / proxy rate")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Patterns by Channel

In [ ]:
channel_summary = (
    df.groupby("ChannelId")
    .agg(
        transactions=(VALUE_COLUMN, "count"),
        avg_value=(VALUE_COLUMN, "mean"),
        proxy_rate=(PROXY_TARGET_COLUMN, "mean"),
    )
    .sort_values("transactions", ascending=False)
)
channel_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

channel_summary["transactions"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Transaction Volume by Channel")
axes[0].set_ylabel("Count")

channel_summary["proxy_rate"].plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Proxy Event Rate by Channel")
axes[1].set_ylabel("Rate")

plt.tight_layout()
plt.show()

## 12. Patterns by Pricing Strategy

In [ ]:
pricing_summary = (
    df.groupby("PricingStrategy")
    .agg(
        transactions=(VALUE_COLUMN, "count"),
        avg_value=(VALUE_COLUMN, "mean"),
        proxy_rate=(PROXY_TARGET_COLUMN, "mean"),
    )
    .sort_values("transactions", ascending=False)
)
pricing_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=pricing_summary.reset_index(),
    x="PricingStrategy",
    y="proxy_rate",
    ax=ax,
)
ax.set_title("Proxy Event Rate by Pricing Strategy")
plt.tight_layout()
plt.show()

## 13. Patterns by Fraud Flag (Proxy Target)

In [ ]:
fraud_counts = df[PROXY_TARGET_COLUMN].value_counts(normalize=True).rename("share")
fraud_counts

In [ ]:
fraud_compare = df.groupby(PROXY_TARGET_COLUMN).agg(
    avg_value=(VALUE_COLUMN, "mean"),
    median_value=(VALUE_COLUMN, "median"),
    avg_amount=(AMOUNT_COLUMN, "mean"),
    transactions=(CUSTOMER_ID_COLUMN, "count"),
)
fraud_compare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x=PROXY_TARGET_COLUMN, ax=axes[0])
axes[0].set_title("Transaction Count by Fraud Flag")

sns.violinplot(data=df, x=PROXY_TARGET_COLUMN, y=VALUE_COLUMN, ax=axes[1], cut=0)
axes[1].set_title("Transaction Value by Fraud Flag")

plt.tight_layout()
plt.show()

## 14. Key Insights Summary

Based on exploration of **95,662 transactions** across **3,742 customers**:

1. **Highly imbalanced proxy target.** Fraud (`FraudResult = 1`) appears in only ~0.2% of transactions and ~1.4% of customers. Any PD/proxy model must address class imbalance (e.g. stratified sampling, class weights, PR-AUC as a primary metric).

2. **Monetary features are heavy-tailed with extreme outliers.** `Amount` and `Value` are strongly correlated (~0.99) and show large standard deviations relative to the median. Log transforms, winsorization, or robust scaling will likely be needed before modeling.

3. **Fraud events concentrate in specific segments.** Transport and certain pricing strategies show materially higher proxy rates than airtime or financial services, suggesting product and pricing context are meaningful risk drivers.

4. **Customer activity is skewed.** The median customer has ~7 transactions while the mean is ~26, indicating a small group of highly active customers dominates volume. Customer-level aggregation will be important for credit scoring.

5. **No missing values in raw data.** Data quality is strong at the transaction level, so feature engineering and customer-level aggregation—not imputation—will be the main preprocessing focus.

---

**Next steps:** Update `NUMERIC_COLUMNS` and `CATEGORICAL_COLUMNS` in `src/config.py`, engineer customer-level features, and validate whether `FraudResult` is an acceptable proxy for default in the Bati Bank business context.